# Bronze to Silver: Life Expectancy

Port the existing World Bank pandas transformation to PySpark. This notebook reads the original CSVs from `raw/`, enriches year-level observations with country metadata, and writes a local Delta table. It is intended for local development before Azure Databricks is available.

In [1]:
from pathlib import Path

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "raw").is_dir() and (path / "silver").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing raw/ and silver/. Open this notebook from the project workspace.")

raw_dir = project_root / "raw"
data_file = raw_dir / "API_SP.DYN.LE00.IN_DS2_en_csv_v2_1259.csv"
country_file = raw_dir / "Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_1259.csv"
output_path = project_root / "silver" / "delta" / "life_expectancy"
baseline_file = project_root / "silver" / "life_expectancy.parquet"

for input_file in (data_file, country_file):
    if not input_file.is_file():
        raise FileNotFoundError(f"Required source file not found: {input_file}")

print(f"Project root: {project_root}")
print(f"World Bank data: {data_file.name}")
print(f"Country metadata: {country_file.name}")
print(f"Delta output: {output_path}")

Project root: c:\Users\ACER\OneDrive\Desktop\Data Engineer Projects Practice
World Bank data: API_SP.DYN.LE00.IN_DS2_en_csv_v2_1259.csv
Country metadata: Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_1259.csv
Delta output: c:\Users\ACER\OneDrive\Desktop\Data Engineer Projects Practice\silver\delta\life_expectancy


## Environment

Run this on a Databricks Runtime, or locally with PySpark and delta-rs installed. Local Spark on Windows avoids Delta's Maven JAR staging (which requires Hadoop winutils); the Delta Rust writer is used for the local output. Databricks continues to use Spark's native Delta writer.

In [2]:
import os
import sys

from pyspark.sql import SparkSession

if os.name == "nt":
    os.environ["PYSPARK_PYTHON"] = sys.executable

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.getActiveSession()
    if spark is None:
        builder = SparkSession.builder.appName("GlobalHealthBronzeToSilver")
        if os.name == "nt":
            builder = builder.master("local[*]")
        spark = builder.getOrCreate()

print(f"Spark version: {spark.version}")

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


## Read the source files

The life-expectancy export has four preamble rows before the CSV header, matching `skiprows=4` in the pandas script.

In [3]:
import csv

with data_file.open(encoding="utf-8-sig", newline="") as source:
    for _ in range(4):
        next(source, None)
    reader = csv.DictReader(source)
    if reader.fieldnames is None:
        raise ValueError("World Bank CSV has no header after its four preamble rows.")
    source_columns = [column for column in reader.fieldnames if column]
    wide_rows = [
        tuple(row.get(column) for column in source_columns)
        for row in reader
    ]
wide_df = spark.createDataFrame(wide_rows, schema=source_columns)
country_metadata_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(str(country_file))
)

print(f"Wide source rows: {wide_df.count():,}")
print(f"Country metadata rows: {country_metadata_raw.count():,}")
wide_df.printSchema()
country_metadata_raw.printSchema()

Wide source rows: 266
Country metadata rows: 278
root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Indicator Name: string (nullable = true)
 |-- Indicator Code: string (nullable = true)
 |-- 1960: string (nullable = true)
 |-- 1961: string (nullable = true)
 |-- 1962: string (nullable = true)
 |-- 1963: string (nullable = true)
 |-- 1964: string (nullable = true)
 |-- 1965: string (nullable = true)
 |-- 1966: string (nullable = true)
 |-- 1967: string (nullable = true)
 |-- 1968: string (nullable = true)
 |-- 1969: string (nullable = true)
 |-- 1970: string (nullable = true)
 |-- 1971: string (nullable = true)
 |-- 1972: string (nullable = true)
 |-- 1973: string (nullable = true)
 |-- 1974: string (nullable = true)
 |-- 1975: string (nullable = true)
 |-- 1976: string (nullable = true)
 |-- 1977: string (nullable = true)
 |-- 1978: string (nullable = true)
 |-- 1979: string (nullable = true)
 |-- 1980: string (nullable = true)
 |-- 1981:

## Reshape year columns

Convert each digit-named year column into a `year` and `value` pair, then discard missing observations.

In [4]:
from pyspark.sql import functions as F

id_columns = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
year_columns = [column for column in wide_df.columns if column.isdigit()]
if not year_columns:
    raise ValueError("No year columns were found in the World Bank CSV. Check the CSV header parsing.")

year_values = F.array(*[
    F.struct(
        F.lit(int(year)).alias("year"),
        F.when(
            F.trim(F.col(year)) != "",
            F.col(year).cast("double"),
        ).otherwise(F.lit(None).cast("double")).alias("value"),
    )
    for year in year_columns
])

long_df = (
    wide_df
    .select(*id_columns, F.explode(year_values).alias("year_value"))
    .select(
        F.col("Country Name").alias("country_name"),
        F.col("Country Code").alias("country_code"),
        F.col("Indicator Name").alias("indicator_name"),
        F.col("Indicator Code").alias("indicator_code"),
        F.col("year_value.year").cast("integer").alias("year"),
        F.col("year_value.value").cast("double").alias("value"),
    )
    .dropna(subset=["value"])
)

print(f"Year columns: {len(year_columns)} ({min(year_columns)}-{max(year_columns)})")
print(f"Non-null country-year observations: {long_df.count():,}")

Year columns: 66 (1960-2025)
Non-null country-year observations: 17,191


## Enrich with country metadata

Keep the same left join and aggregate definition as the local pandas transformation.

In [5]:
country_metadata = (
    country_metadata_raw
    .select(
        F.col("Country Code").alias("country_code"),
        F.col("Region").alias("region"),
        F.col("IncomeGroup").alias("income_group"),
    )
)

silver_df = (
    long_df
    .join(country_metadata, on="country_code", how="left")
    .withColumn(
        "is_aggregate",
        F.col("region").isNull() | (F.trim(F.col("region")) == ""),
    )
    .select(
        "country_name",
        "country_code",
        "indicator_name",
        "indicator_code",
        "year",
        "value",
        "region",
        "income_group",
        "is_aggregate",
    )
)

silver_df.printSchema()

root
 |-- country_name: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- indicator_name: string (nullable = true)
 |-- indicator_code: string (nullable = true)
 |-- year: integer (nullable = false)
 |-- value: double (nullable = true)
 |-- region: string (nullable = true)
 |-- income_group: string (nullable = true)
 |-- is_aggregate: boolean (nullable = true)



## Write Delta silver

This local path is separate from the existing Parquet baseline. Locally, delta-rs writes the Spark result as Delta without staging Maven JARs. In Databricks, Spark's native Delta writer writes the output.

In [6]:
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .save(str(output_path))
    )
else:
    from deltalake import write_deltalake

    arrow_table = silver_df.toArrow()
    write_deltalake(str(output_path), arrow_table, mode="overwrite")

print(f"Wrote Delta silver data to: {output_path}")

Wrote Delta silver data to: c:\Users\ACER\OneDrive\Desktop\Data Engineer Projects Practice\silver\delta\life_expectancy


## Validate the output

Check required columns and values, print row and year statistics, and compare the row count with the existing pandas-generated Parquet file when available.

In [7]:
expected_columns = [
    "country_name", "country_code", "indicator_name", "indicator_code",
    "year", "value", "region", "income_group", "is_aggregate",
]
assert silver_df.columns == expected_columns, f"Unexpected columns: {silver_df.columns}"

validation = silver_df.agg(
    F.count("*").alias("row_count"),
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year"),
    F.sum(F.when(F.col("is_aggregate"), 1).otherwise(0)).alias("aggregate_rows"),
    F.sum(F.when(~F.col("is_aggregate"), 1).otherwise(0)).alias("country_rows"),
    F.sum(F.when(F.col("year").isNull() | F.col("value").isNull(), 1).otherwise(0)).alias("null_year_or_value"),
).first()

assert validation.row_count > 0, "Silver output is empty."
assert validation.null_year_or_value == 0, "Silver output contains a null year or value."

print(f"Silver rows: {validation.row_count:,}")
print(f"Year range: {validation.min_year}-{validation.max_year}")
print(f"Country rows: {validation.country_rows:,}")
print(f"Aggregate rows: {validation.aggregate_rows:,}")

if baseline_file.is_file():
    baseline_count = spark.read.parquet(str(baseline_file)).count()
    print(f"Existing pandas silver rows: {baseline_count:,}")
    assert validation.row_count == baseline_count, (
        f"Row-count mismatch: Spark={validation.row_count:,}, pandas={baseline_count:,}"
    )
    print("Row-count comparison passed.")
else:
    print(f"Baseline Parquet not found; skipped count comparison: {baseline_file}")

Silver rows: 17,191
Year range: 1960-2024
Country rows: 14,071
Aggregate rows: 3,120
Existing pandas silver rows: 17,191
Row-count comparison passed.
